# El costo invisible del progreso
### ¿Cambia la identificación de los países que desacoplan su crecimiento de las emisiones de CO₂ al pasar de la contabilidad territorial a la basada en consumo?

**Proyecto 2 · DATA-1002 Aplicaciones en Ciencia de Datos**

Integrantes: _(completar)_

**Usuario del análisis:** comité técnico de desarrollo sostenible y transición climática.
**Decisión que apoya:** qué países estudiar como referentes de desacoplamiento y cuáles requieren un diagnóstico adicional porque su avance territorial no se sostiene al considerar el comercio.
**Unidad de análisis:** país-año. **Período común:** 1990–2023.

| Sección | Contenido |
|---|---|
| 1 | Carga e inspección de las cuatro fuentes |
| 2 | Limpieza: países vs. agregados, nombres de columnas |
| 3 | Integración y verificación de cardinalidad |
| 4 | Variables derivadas (emisiones totales, PIB per cápita, intensidad, brecha consumo–territorial) |
| 5 | Clasificación del desacoplamiento (territorial y consumo) |
| 6 | ¿Cuántos países cambian de clase? (respuesta a la pregunta) |
| 7 | Per cápita e intensidad |
| 8 | Robustez con otro período |
| 9 | Exportación para el dashboard y limitaciones |

In [ ]:
import io
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

CARPETA = Path(".")                      # carpeta del proyecto (donde están los .zip)
SALIDA = CARPETA / "datos_procesados"
SALIDA.mkdir(exist_ok=True)

## 1. Carga e inspección de las fuentes

| Fuente | Productor original | Archivo | Unidad |
|---|---|---|---|
| CO₂ territorial per cápita | Global Carbon Budget (2026), procesado por Our World in Data | `co2-emissions-per-capita.zip` | t CO₂ por persona |
| CO₂ basado en consumo per cápita | Global Carbon Budget (2026), procesado por Our World in Data | `per-capita-consumption-based-co2-emissions.zip` | t CO₂ por persona |
| PIB real (PPA) | Banco Mundial, Eurostat, OCDE y FMI (2026), vía Our World in Data | `gdp-worldbank.zip` | dólares internacionales de 2021 |
| Población | HYDE (2023), Gapminder (2022), UN WPP (2024), vía Our World in Data | `population.csv` | personas |

Licencia de Our World in Data: CC BY 4.0 (se debe citar la fuente). Los archivos de OWID vienen en ZIP con el CSV, un `readme.md` y un `.metadata.json`; se leen directamente desde el ZIP, sin descomprimir a mano.

**¿Por qué cuatro fuentes?** Las dos series de CO₂ vienen *per cápita* y el PIB viene en *total*. Para comparar crecimiento económico con emisiones se necesita la población, que convierte las emisiones per cápita en totales.

In [ ]:
def leer_owid_zip(ruta_zip):
    """Lee el CSV que viene dentro de un ZIP descargado de Our World in Data."""
    with zipfile.ZipFile(ruta_zip) as z:
        nombre_csv = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(nombre_csv) as f:
            return pd.read_csv(f)

co2_terr = leer_owid_zip(CARPETA / "co2-emissions-per-capita.zip")
co2_cons = leer_owid_zip(CARPETA / "per-capita-consumption-based-co2-emissions.zip")
pib = leer_owid_zip(CARPETA / "gdp-worldbank.zip")

La población se descarga una sola vez desde Our World in Data con `requests` (como en la Sesión 9) y se guarda en `population.csv`. Si el archivo ya existe, se usa el guardado. El archivo incluido en la carpeta contiene los 120 países con datos de consumo, 1990–2023.

In [ ]:
RUTA_POBLACION = CARPETA / "population.csv"
URL_POBLACION = "https://ourworldindata.org/grapher/population.csv?v=1&csvType=full&useColumnShortNames=false"

if not RUTA_POBLACION.exists():
    import requests
    r = requests.get(URL_POBLACION, timeout=60)
    print("Estado de la descarga:", r.status_code)
    r.raise_for_status()
    RUTA_POBLACION.write_bytes(r.content)

poblacion = pd.read_csv(RUTA_POBLACION)
poblacion.head()

In [ ]:
def inspeccionar(nombre, df):
    col_valor = df.columns[-1]
    codigos = df["Code"].astype("string").fillna("")
    return {
        "fuente": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "años": f"{df['Year'].min()}–{df['Year'].max()}",
        "entidades": df["Code"].nunique(),
        "sin código": int(df["Code"].isna().sum()),
        "agregados (OWID_ / WB_)": int(codigos.str.match(r"^(OWID_|WB_)").sum()),
        "duplicados código-año": int(df.duplicated(["Code", "Year"]).sum()),
        "nulos en el valor": int(df[col_valor].isna().sum()),
    }

pd.DataFrame([
    inspeccionar("CO₂ territorial", co2_terr),
    inspeccionar("CO₂ consumo", co2_cons),
    inspeccionar("PIB", pib),
    inspeccionar("Población", poblacion),
]).set_index("fuente")

**Lectura:**
- En todas las fuentes, **una fila = una entidad en un año**. No hay duplicados código-año ni valores nulos.
- Las fuentes de OWID mezclan países con **agregados** (continentes, grupos de ingreso, regiones del Banco Mundial como `WB_LAC`, o entidades sin código como "Europe (excl. EU-27)"). Si se dejan, un país se contaría dos veces dentro de su región.
- Las emisiones de consumo solo cubren 1990–2025 y unos 120 países; las territoriales llegan desde 1750.

## 2. Limpieza

Decisiones:
1. Conservar solo países: códigos ISO alfa-3 de tres letras mayúsculas. Esto elimina agregados de OWID (`OWID_*`), del Banco Mundial (`WB_*`) y entidades sin código.
2. Renombrar columnas a nombres cortos en español.
3. Restringir al período común de las cuatro fuentes.

In [ ]:
ES_PAIS = r"^[A-Z]{3}$"

def solo_paises(df, nombre_valor):
    codigos = df["Code"].astype("string").fillna("")
    out = df[codigos.str.match(ES_PAIS)].copy()
    out = out.rename(columns={"Code": "iso3", "Year": "anio", out.columns[-1]: nombre_valor})
    return out

terr = solo_paises(co2_terr, "co2_terr_pc").rename(columns={"Entity": "pais"})
cons = solo_paises(co2_cons, "co2_cons_pc").drop(columns="Entity")
gdp = solo_paises(pib, "pib").drop(columns="Entity")
pob = poblacion.rename(columns={"Code": "iso3", "Year": "anio", "Population": "poblacion"})
pob = pob[pob["iso3"].astype("string").fillna("").str.match(ES_PAIS)][["iso3", "anio", "poblacion"]]

for nombre, df in [("territorial", terr), ("consumo", cons), ("PIB", gdp), ("población", pob)]:
    print(f"{nombre:12s} países: {df['iso3'].nunique():4d}  años: {df['anio'].min()}–{df['anio'].max()}  filas: {len(df)}")

## 3. Integración

La llave es **`iso3` + `anio`**. Primero se une con `how="outer"` e `indicator=True` para ver qué registros no tienen pareja; luego se conserva el período y los países con las cuatro variables.

`validate="one_to_one"` garantiza que ninguna llave se repita: si alguna se repitiera, la unión multiplicaría filas y pandas lanzaría un error.

In [ ]:
# Paso 1: CO2 de consumo (la fuente más restrictiva) con el PIB, para auditar la unión
auditoria = cons.merge(gdp, on=["iso3", "anio"], how="outer", validate="one_to_one", indicator=True)
print(auditoria["_merge"].value_counts().rename({"left_only": "solo consumo", "right_only": "solo PIB", "both": "ambos"}))

sin_pib = sorted(auditoria.loc[auditoria["_merge"] == "left_only", "iso3"].unique())
print("\nPaíses con emisiones de consumo pero sin PIB:", sin_pib)

In [ ]:
# Paso 2: unión completa de las cuatro fuentes
panel = (
    terr.merge(cons, on=["iso3", "anio"], how="inner", validate="one_to_one")
        .merge(gdp, on=["iso3", "anio"], how="inner", validate="one_to_one")
        .merge(pob, on=["iso3", "anio"], how="inner", validate="one_to_one")
)

assert not panel.duplicated(["iso3", "anio"]).any()
assert panel[["co2_terr_pc", "co2_cons_pc", "pib", "poblacion"]].notna().all().all()

print("Filas del panel integrado:", len(panel))
print("Países:", panel["iso3"].nunique())
print("Años:", panel["anio"].min(), "–", panel["anio"].max())
print("Observaciones por país:", panel.groupby("iso3").size().describe()[["min", "50%", "max"]].to_dict())

**Resultado de la unión:** cada fila del panel es un país-año con las cuatro variables. El año final es 2023 porque la serie de población llega hasta ese año. Taiwán y Venezuela quedan fuera porque no tienen PIB del Banco Mundial; los países sin emisiones de consumo (casi todos pequeños o de bajos ingresos) también quedan fuera. Esto es una **limitación de cobertura** que se debe mostrar en el dashboard.

## 4. Variables derivadas

| Variable | Fórmula | Unidad |
|---|---|---|
| `co2_terr_mt` | CO₂ territorial per cápita × población / 10⁶ | millones de t CO₂ |
| `co2_cons_mt` | CO₂ de consumo per cápita × población / 10⁶ | millones de t CO₂ |
| `pib_pc` | PIB / población | dólares internacionales de 2021 por persona |
| `intensidad_terr` | CO₂ territorial / PIB | kg CO₂ por dólar |
| `brecha_cons_pct` | (consumo − territorial) / territorial × 100 | % |
| `balance_comercio` | "importador neto" si consumo > territorial, si no "exportador neto" | categoría |

In [ ]:
panel["co2_terr_mt"] = panel["co2_terr_pc"] * panel["poblacion"] / 1e6
panel["co2_cons_mt"] = panel["co2_cons_pc"] * panel["poblacion"] / 1e6
panel["pib_pc"] = panel["pib"] / panel["poblacion"]
panel["intensidad_terr"] = panel["co2_terr_mt"] * 1e9 / panel["pib"]      # kg por dólar
panel["intensidad_cons"] = panel["co2_cons_mt"] * 1e9 / panel["pib"]
panel["brecha_cons_pct"] = (panel["co2_cons_pc"] - panel["co2_terr_pc"]) / panel["co2_terr_pc"] * 100
panel["balance_comercio"] = np.where(panel["co2_cons_pc"] > panel["co2_terr_pc"], "Importador neto", "Exportador neto")

panel.sort_values(["iso3", "anio"]).head()

### Control de calidad: emisiones de consumo que son copia de las territoriales

En algunos países y años la serie de consumo es **idéntica** a la territorial. No es un resultado real: el Global Carbon Budget no tenía datos de comercio para esos años y repitió el valor territorial. Si no se corrige, el "cambio" de las emisiones de consumo de esos países sería un artefacto.

In [ ]:
panel["consumo_es_copia"] = np.isclose(panel["co2_cons_pc"], panel["co2_terr_pc"], rtol=0, atol=1e-9)

copias = (panel[panel["consumo_es_copia"]]
          .groupby(["iso3", "pais"])["anio"].agg(desde="min", hasta="max", años="count"))
copias

**Decisión:** esos países no se clasifican con emisiones de consumo cuando alguna copia cae dentro de la ventana inicial. Quedan como *Sin dato confiable de consumo* y se excluyen de la tabla cruzada. Su clasificación territorial sí se conserva.

## 5. Clasificación del desacoplamiento

Se comparan dos ventanas de cinco años para no depender de un año atípico (la crisis de 2008–2009 o la pandemia de 2020):

- **Período inicial:** promedio 1995–1999
- **Período final:** promedio 2019–2023

Para cada país se calcula el cambio porcentual del PIB y de las emisiones totales entre ambos períodos, y se clasifica así (definición usual de la OCDE):

| Clase | Regla |
|---|---|
| Desacoplamiento absoluto | El PIB crece y las emisiones caen |
| Desacoplamiento relativo | El PIB crece y las emisiones crecen menos que el PIB |
| Sin desacoplamiento | El PIB crece y las emisiones crecen igual o más que el PIB |
| PIB sin crecimiento | El PIB no creció (fuera del alcance de la pregunta) |

La clasificación se hace dos veces: con emisiones territoriales y con emisiones de consumo.

In [ ]:
INICIO = (1995, 1999)
FINAL = (2019, 2023)
SIN_DATO = "Sin dato confiable de consumo"
ORDEN_CLASES = ["Desacoplamiento absoluto", "Desacoplamiento relativo", "Sin desacoplamiento", "PIB sin crecimiento", SIN_DATO]

def clasificar(g_pib, g_co2):
    if g_pib <= 0:
        return "PIB sin crecimiento"
    if g_co2 < 0:
        return "Desacoplamiento absoluto"
    if g_co2 < g_pib:
        return "Desacoplamiento relativo"
    return "Sin desacoplamiento"

def tabla_cambios(panel, inicio, final, columnas=("pib", "co2_terr_mt", "co2_cons_mt")):
    """Promedia cada ventana y calcula el cambio % entre ambas. Exige los 5 años en cada ventana."""
    def ventana(a, b):
        v = panel[panel["anio"].between(a, b)]
        completos = v.groupby("iso3")["anio"].nunique()
        v = v[v["iso3"].isin(completos[completos == (b - a + 1)].index)]
        return v.groupby("iso3")[list(columnas)].mean()

    v0, v1 = ventana(*inicio), ventana(*final)
    comunes = v0.index.intersection(v1.index)
    cambio = (v1.loc[comunes] / v0.loc[comunes] - 1) * 100
    cambio.columns = ["g_" + c for c in columnas]
    nombres = panel.drop_duplicates("iso3").set_index("iso3")["pais"]
    cambio.insert(0, "pais", nombres.loc[comunes])
    return cambio

cambios = tabla_cambios(panel, INICIO, FINAL)
cambios["clase_territorial"] = [clasificar(p, c) for p, c in zip(cambios["g_pib"], cambios["g_co2_terr_mt"])]
cambios["clase_consumo"] = [clasificar(p, c) for p, c in zip(cambios["g_pib"], cambios["g_co2_cons_mt"])]

def marcar_sin_dato(tabla, inicio, columna):
    """Anula la clase de consumo de los países con datos copiados en la ventana inicial."""
    malos = panel.loc[panel["anio"].between(*inicio) & panel["consumo_es_copia"], "iso3"].unique()
    tabla.loc[tabla.index.isin(malos), columna] = SIN_DATO
    return tabla

cambios = marcar_sin_dato(cambios, INICIO, "clase_consumo")
validos = cambios[cambios["clase_consumo"] != SIN_DATO].copy()
validos["cambia_de_clase"] = validos["clase_territorial"] != validos["clase_consumo"]
cambios["cambia_de_clase"] = validos["cambia_de_clase"]

fuera = sorted(set(panel["iso3"]) - set(cambios.index))
print("Países clasificados:", len(cambios))
print("Sin los 5 años de la ventana inicial (no se clasifican):", fuera)
print("Con dato de consumo copiado en la ventana inicial:", sorted(cambios.index[cambios["clase_consumo"] == SIN_DATO]))
print("Países con ambas clasificaciones válidas:", len(validos))
cambios.round(1).sort_values("g_co2_terr_mt").head(10)

In [ ]:
# Solo los países con ambas clasificaciones válidas, para comparar lo mismo
resumen = pd.DataFrame({
    "Territorial": validos["clase_territorial"].value_counts(),
    "Consumo": validos["clase_consumo"].value_counts(),
}).reindex(ORDEN_CLASES[:4]).fillna(0).astype(int)
resumen.loc["Total"] = resumen.sum()
resumen

## 6. ¿Cuántos países cambian de clase al pasar a la contabilidad de consumo?

Esta tabla cruzada responde directamente la pregunta analítica: la diagonal son los países cuya clase se mantiene y lo que está fuera de ella son los que cambian.

In [ ]:
cruce = pd.crosstab(validos["clase_territorial"], validos["clase_consumo"],
                    rownames=["Clase territorial"], colnames=["Clase de consumo"])
cruce = cruce.reindex(index=ORDEN_CLASES[:4], columns=ORDEN_CLASES[:4], fill_value=0)

con_crecimiento = validos[validos["clase_territorial"] != "PIB sin crecimiento"]
n_cambian = int(con_crecimiento["cambia_de_clase"].sum())
print(f"Países con crecimiento del PIB: {len(con_crecimiento)}")
print(f"Cambian de clase al usar consumo: {n_cambian} ({n_cambian / len(con_crecimiento):.0%})")
cruce

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
m = cruce.loc[ORDEN_CLASES[:3], ORDEN_CLASES[:3]]
ax.imshow(m.values, cmap="Blues")
etiquetas = ["Absoluto", "Relativo", "Sin desacoplamiento"]
ax.set_xticks(range(3), etiquetas)
ax.set_yticks(range(3), etiquetas)
for i in range(3):
    for j in range(3):
        v = m.values[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=13,
                color="white" if v > m.values.max() / 2 else "black")
ax.set_xlabel("Clase con emisiones de consumo")
ax.set_ylabel("Clase con emisiones territoriales")
ax.set_title(f"Clasificación del desacoplamiento, {INICIO[0]}–{INICIO[1]} vs {FINAL[0]}–{FINAL[1]}\n(número de países)")
plt.tight_layout()
plt.show()

### Los casos que más importan para el comité

- **Absoluto en territorio, pero no en consumo:** su reducción territorial no se sostiene cuando se suman las emisiones importadas. Son candidatos a **diagnóstico adicional**.
- **Absoluto en ambas contabilidades:** reducen emisiones sin importar cómo se midan. Son candidatos a **referentes**.

In [ ]:
cols = ["pais", "g_pib", "g_co2_terr_mt", "g_co2_cons_mt", "clase_territorial", "clase_consumo"]

referentes = validos[(validos["clase_territorial"] == "Desacoplamiento absoluto") &
                     (validos["clase_consumo"] == "Desacoplamiento absoluto")]
diagnostico = validos[(validos["clase_territorial"] == "Desacoplamiento absoluto") &
                      (validos["clase_consumo"] != "Desacoplamiento absoluto")]

print(f"Referentes (absoluto en ambas): {len(referentes)}")
print(f"Requieren diagnóstico (absoluto solo en territorio): {len(diagnostico)}\n")
diagnostico[cols].round(1).sort_values("g_co2_cons_mt", ascending=False)

In [ ]:
referentes[cols].round(1).sort_values("g_co2_cons_mt")

In [ ]:
# Dispersión: cambio de emisiones territoriales vs de consumo
colores = {"Desacoplamiento absoluto": "#2a9d8f", "Desacoplamiento relativo": "#e9c46a",
           "Sin desacoplamiento": "#e76f51", "PIB sin crecimiento": "#999999"}

fig, ax = plt.subplots(figsize=(8, 6))
for clase, grupo in validos.groupby("clase_territorial"):
    ax.scatter(grupo["g_co2_terr_mt"], grupo["g_co2_cons_mt"], s=30, alpha=0.8,
               color=colores[clase], label=f"{clase} (territorial)")
lim = [-60, 340]
ax.plot(lim, lim, color="grey", lw=1, ls="--", label="Igual cambio en ambas")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
for iso in diagnostico.index:
    ax.annotate(iso, (validos.loc[iso, "g_co2_terr_mt"], validos.loc[iso, "g_co2_cons_mt"]), fontsize=8)
ax.set_xlim(-60, 340)
ax.set_ylim(-60, 340)
ax.set_xlabel("Cambio % de emisiones territoriales")
ax.set_ylabel("Cambio % de emisiones de consumo")
ax.set_title("Cambio de emisiones 1995–1999 vs 2019–2023\nArriba de la diagonal: el CO₂ de consumo creció más que el territorial")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

### Trayectorias de países ilustrativos

Índice 1995 = 100 para el PIB, las emisiones territoriales y las de consumo. La distancia entre las dos líneas de CO₂ muestra cuánto pesa el comercio.

In [ ]:
def trayectoria(iso, ax):
    d = panel[(panel["iso3"] == iso) & (panel["anio"] >= 1995)].set_index("anio")
    base = d.loc[1995]
    ax.plot(d.index, d["pib"] / base["pib"] * 100, label="PIB", color="#264653", lw=2)
    ax.plot(d.index, d["co2_terr_mt"] / base["co2_terr_mt"] * 100, label="CO₂ territorial", color="#e76f51")
    ax.plot(d.index, d["co2_cons_mt"] / base["co2_cons_mt"] * 100, label="CO₂ consumo", color="#2a9d8f", ls="--")
    ax.axhline(100, color="grey", lw=0.6)
    ax.set_title(d["pais"].iloc[0])

# Cambien esta lista por los casos que quieran mostrar
ejemplos = ["GBR", "USA", "CHN", "COL"]
fig, axes = plt.subplots(1, len(ejemplos), figsize=(16, 4), sharey=False)
for iso, ax in zip(ejemplos, axes):
    trayectoria(iso, ax)
axes[0].set_ylabel("Índice (1995 = 100)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### ¿Importadores o exportadores netos de emisiones?

Si un país es importador neto (consumo > territorial), parte de las emisiones de lo que consume ocurren en otros países. Una brecha que **crece** en el tiempo sugiere que la reducción territorial podría deberse en parte a trasladar producción.

In [ ]:
brecha = (panel[panel["anio"].isin([1995, 2023])]
          .pivot_table(index="iso3", columns="anio", values="brecha_cons_pct"))
brecha.columns = ["brecha_1995", "brecha_2023"]
brecha["cambio_brecha_pp"] = brecha["brecha_2023"] - brecha["brecha_1995"]
brecha = brecha.join(cambios[["pais", "clase_territorial", "clase_consumo"]], how="inner")

print("Importadores netos en 2023:", int((brecha["brecha_2023"] > 0).sum()), "de", len(brecha))
brecha.round(1).sort_values("brecha_2023", ascending=False).head(12)

## 7. ¿Cambia la interpretación con emisiones per cápita o por unidad de PIB?

- **Per cápita:** se repite la clasificación con PIB per cápita y CO₂ per cápita. Quita el efecto del crecimiento de la población.
- **Intensidad:** kg de CO₂ por dólar de PIB. Si cae, la economía es más eficiente en carbono, pero eso por sí solo solo implica desacoplamiento relativo.

In [ ]:
cambios_pc = tabla_cambios(panel, INICIO, FINAL, columnas=("pib_pc", "co2_terr_pc", "co2_cons_pc"))
cambios_pc["clase_territorial_pc"] = [clasificar(p, c) for p, c in zip(cambios_pc["g_pib_pc"], cambios_pc["g_co2_terr_pc"])]
cambios_pc["clase_consumo_pc"] = [clasificar(p, c) for p, c in zip(cambios_pc["g_pib_pc"], cambios_pc["g_co2_cons_pc"])]
cambios_pc = marcar_sin_dato(cambios_pc, INICIO, "clase_consumo_pc")

comparacion = pd.DataFrame({
    "Totales · territorial": cambios["clase_territorial"].value_counts(),
    "Totales · consumo": cambios["clase_consumo"].value_counts(),
    "Per cápita · territorial": cambios_pc["clase_territorial_pc"].value_counts(),
    "Per cápita · consumo": cambios_pc["clase_consumo_pc"].value_counts(),
}).reindex(ORDEN_CLASES).fillna(0).astype(int)
comparacion

In [ ]:
intensidad = tabla_cambios(panel, INICIO, FINAL, columnas=("intensidad_terr", "intensidad_cons"))
intensidad = intensidad[intensidad.index.isin(validos.index)]
print("Países cuya intensidad territorial cayó:", int((intensidad["g_intensidad_terr"] < 0).sum()), "de", len(intensidad))
print("Países cuya intensidad de consumo cayó:", int((intensidad["g_intensidad_cons"] < 0).sum()), "de", len(intensidad))
print("Mediana del cambio de intensidad territorial: %.1f %%" % intensidad["g_intensidad_terr"].median())
print("Mediana del cambio de intensidad de consumo: %.1f %%" % intensidad["g_intensidad_cons"].median())

## 8. Robustez: ¿se sostiene con otro período?

Se repite la clasificación con una ventana inicial más reciente (2005–2009). Si las cifras principales cambian mucho, la conclusión depende del período elegido y hay que decirlo en la presentación.

In [ ]:
def resumen_periodo(inicio, final):
    c = tabla_cambios(panel, inicio, final)
    c["t"] = [clasificar(p, x) for p, x in zip(c["g_pib"], c["g_co2_terr_mt"])]
    c["k"] = [clasificar(p, x) for p, x in zip(c["g_pib"], c["g_co2_cons_mt"])]
    c = marcar_sin_dato(c, inicio, "k")
    c = c[c["k"] != SIN_DATO]
    crece = c[c["t"] != "PIB sin crecimiento"]
    return {
        "período": f"{inicio[0]}–{inicio[1]} vs {final[0]}–{final[1]}",
        "países válidos": len(c),
        "absoluto territorial": int((c["t"] == "Desacoplamiento absoluto").sum()),
        "absoluto consumo": int((c["k"] == "Desacoplamiento absoluto").sum()),
        "absoluto en ambas": int(((c["t"] == "Desacoplamiento absoluto") & (c["k"] == "Desacoplamiento absoluto")).sum()),
        "cambian de clase": int((crece["t"] != crece["k"]).sum()),
    }

pd.DataFrame([resumen_periodo(INICIO, FINAL), resumen_periodo((2005, 2009), FINAL)]).set_index("período")

## 9. Exportación para el dashboard

El dashboard se alimenta de estos dos archivos (datos integrados, no de las fuentes por separado):
- `datos_procesados/panel_integrado.csv`: un registro por país-año con todas las variables.
- `datos_procesados/clasificacion_paises.csv`: un registro por país con los cambios y las clases.

In [ ]:
panel.sort_values(["iso3", "anio"]).to_csv(SALIDA / "panel_integrado.csv", index=False)

clasificacion = (cambios.join(cambios_pc[["g_pib_pc", "g_co2_terr_pc", "g_co2_cons_pc", "clase_territorial_pc", "clase_consumo_pc"]])
                        .join(brecha[["brecha_1995", "brecha_2023", "cambio_brecha_pp"]]))
clasificacion.reset_index().to_csv(SALIDA / "clasificacion_paises.csv", index=False)

print(f"panel_integrado.csv: {len(panel)} filas")
print(f"clasificacion_paises.csv: {len(clasificacion)} países")

## Limitaciones

1. **Cobertura:** las emisiones de consumo existen para unos 120 países con datos de comercio suficientes; el análisis no cubre a muchos países pequeños o de bajos ingresos. Taiwán y Venezuela quedan fuera por falta de PIB.
2. **Alcance de las emisiones:** ambas series incluyen solo CO₂ fósil e industrial. No incluyen cambio de uso del suelo (deforestación) ni aviación y navegación internacional, ni otros gases de efecto invernadero.
3. **Emisiones de consumo estimadas:** se calculan con modelos insumo-producto y tienen más incertidumbre que las territoriales. En cinco países (Bélgica, Luxemburgo, Sudáfrica, Namibia y Botsuana) la serie de consumo de los años 90 es una copia de la territorial, así que no se clasificaron con consumo en el período 1995–1999. Noruega no tiene datos de consumo antes de 2003.
4. **Definición de períodos:** la clasificación depende de las ventanas elegidas (sección 8).
5. **Sin causalidad:** el análisis es descriptivo. Un cambio de clase es compatible con traslado de producción al exterior, pero no lo demuestra.
6. **Fuente intermedia:** las cuatro series se descargaron de Our World in Data, que procesa datos de otros productores; se citan los productores originales.

## Fuentes

- Global Carbon Budget (2026); Population based on various sources (2024) – with major processing by Our World in Data. *CO₂ emissions per capita* y *Per capita consumption-based CO₂ emissions*. https://ourworldindata.org/co2-emissions
- Eurostat, OECD, IMF, and World Bank (2026) – with minor processing by Our World in Data. *Gross domestic product (GDP)*. https://ourworldindata.org/grapher/gdp-worldbank
- HYDE (2023); Gapminder (2022); UN WPP (2024) – with major processing by Our World in Data. *Population*. https://ourworldindata.org/grapher/population